# Environment maintenance
Use this notebook to (1) check the generator's output with the HEALTH CHECK cell before your first pipeline update and (2) reset the lab: drop the tables, then remove the raw files.

**Run cells one at a time. Never Run all:** the reset cells further down drop your tables and delete the generator's files.

After a reset, run `data_generator.py` again (it backfills again, because the sales table is empty), then start the pipeline with **Run pipeline with full table refresh** so every table rebuilds from the new landing data.

### HEALTH CHECK
The `sales_count`, `customers_count`, `items_count` and `stores_count` rows show how many raw records each landing table holds (`items_count` is the products); all four must be above 0.

The `missing_customers`, `missing_products` and `missing_stores` rows check referential integrity: sales whose customer, product or store is missing from its lookup table (all should be 0).

In [0]:
from variables import *

display(spark.sql(f"""
SELECT "sales_count" as table_name, count(*) as record_count FROM delta.`{RAW_SALES_PATH}`
UNION 
SELECT "customers_count", count(*) FROM delta.`{RAW_CUSTOMERS_PATH}`
UNION
SELECT "items_count", count(*) FROM delta.`{RAW_PRODUCTS_PATH}`
UNION
SELECT "stores_count", count(*) FROM delta.`{RAW_STORES_PATH}`
UNION
SELECT "missing_customers", COUNT(*)
FROM delta.`{RAW_SALES_PATH}` s
LEFT ANTI JOIN delta.`{RAW_CUSTOMERS_PATH}` c
ON s.customer_id = c.customer_id
UNION
SELECT "missing_products", COUNT(*)
FROM delta.`{RAW_SALES_PATH}` s
LEFT ANTI JOIN delta.`{RAW_PRODUCTS_PATH}` i
ON s.product_id = i.product_id
UNION
SELECT "missing_stores", COUNT(*)
FROM delta.`{RAW_SALES_PATH}` s
LEFT ANTI JOIN delta.`{RAW_STORES_PATH}` st
ON s.store_id = st.store_id;
"""))

### List the files the generator wrote

In [0]:
for path in [RAW_SALES_PATH, RAW_CUSTOMERS_PATH, RAW_PRODUCTS_PATH, RAW_STORES_PATH]:
    display(dbutils.fs.ls(path)[:10])

## Reset the lab
Run these only to start over. They drop every table in `01_bronze`, `02_silver` and `03_gold`, then delete the generator's files. The import below loads the names, so you can run this section on its own.

In [0]:
from variables import *

### Drop all tables in the bronze, silver and gold schemas

In [0]:
print("--- Starting to drop tables ---")
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
  full_schema_name = f"{CATALOG_NAME}.{schema}"
  try:
    print(f"\nProcessing schema: {full_schema_name}")
    # Get all tables in the current catalog and schema
    tables_df = spark.sql(f"SHOW TABLES IN {full_schema_name}")
    
    # Collect the names first, then drop
    tables_to_check = [row.tableName for row in tables_df.collect()]

    for table_name in tables_to_check:
      full_table_name = f"{full_schema_name}.{table_name}"
      try:
          spark.sql(f"DROP TABLE {full_table_name}")
          print(f"  SUCCESS: Dropped {full_table_name}")
      except Exception as e:
          print(f"  FAILED to drop {full_table_name}: {e}")

  except Exception as e:
    print(f"Could not process schema '{full_schema_name}': {e}")

print("\n--- Deletion process complete ---")


### Remove all raw files from the data generator

In [0]:
for path in [RAW_SALES_PATH, RAW_CUSTOMERS_PATH, RAW_PRODUCTS_PATH, RAW_STORES_PATH]:
    dbutils.fs.rm(path, recurse=True)